In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_01_contacts_all
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_01_contacts_all
# Spec Ref  : §1.2 Refresh Materialized Views
# Purpose   : Build contacts_all Delta table = contacts UNION ALL crm_contacts
#             (currently just hgi.silver.contacts — ready for multi-source union)
#             This MUST run FIRST before any other map notebook.
#             All downstream map queries join against contacts_all.
#
# Why Delta table instead of plain VIEW?
#   A plain VIEW re-scans hgi.silver.contacts on every downstream JOIN.
#   A Delta table is read once, cached on disk, and skipped by Liquid Clustering.
#   For 100k+ contacts this is the difference between seconds and minutes.
#
# Serverless: works on 2X-Small — safe_spark_conf skips unsupported configs
# Job Compute: all SPARK_CONF_MAP configs apply for full performance
# =============================================================================

In [0]:

# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)

print(f"=== Map 01: contacts_all ===  customer={customer_id}  tenant={tenant_id}")
print(f"  Reading from : {sv}.contacts")
print(f"  Writing to   : {sv}.contacts_all")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

initialize_audit_tables()

In [0]:
# CELL 3 ── CDF-Aware Incremental MERGE for contacts_all (tenant-scoped)
# FIX v2: Use SOURCE table version (stored as property) instead of TARGET history
# This prevents version mismatch errors when target has more versions than source
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.contacts_all")

    # Ensure table exists (idempotent)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {sv}.contacts_all (
            id                    STRING NOT NULL,
            tenant                BIGINT,
            email                 STRING,
            domain                STRING,
            source_system         STRING,
            source_system_object  STRING,
            source_key_name       STRING,
            source_key_value      STRING,
            data_timestamp        TIMESTAMP,
            a_accountid           STRING
        )
        USING DELTA
        CLUSTER BY (tenant, email, domain)
        {DELTA_TBLPROPS_MAP}
    """)

    # ── VERSION TRACKING (v2): Read last-processed SOURCE version from target property ──
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {sv}.contacts_all ('last_source_cdf_version')").collect()
        val = props[0][1] if props else None
        last_source_ver = int(val) if val and val != '(not specified)' else 0
    except Exception:
        last_source_ver = 0

    # Get current max version of the SOURCE table (silver.contacts)
    try:
        source_max_ver = spark.sql(f"SELECT MAX(version) FROM (DESCRIBE HISTORY {sv}.contacts)").collect()[0][0] or 0
    except Exception:
        source_max_ver = 0

    print(f"  CDF: source={sv}.contacts  last_read_ver={last_source_ver}  current_source_ver={source_max_ver}")

    # Check if this is first run (contacts_all is empty) — do full load
    target_count = spark.sql(f"SELECT COUNT(*) FROM {sv}.contacts_all WHERE tenant = {tenant_id}").collect()[0][0]
    
    if target_count == 0:
        # First run: full load from source
        print(f"  First run detected — loading all contacts for tenant {tenant_id}")
        source_df = spark.sql(f"""
            SELECT id, tenant, email, domain, source_system, source_system_object,
                   source_key_name, source_key_value, data_timestamp, a_accountid
            FROM {sv}.contacts
            WHERE id IS NOT NULL AND tenant = {tenant_id}
        """)
        cdf_count = source_df.count()
    elif last_source_ver >= source_max_ver:
        # No new versions on source — nothing to process
        print(f"  Source table unchanged (ver {source_max_ver}) — skipping")
        cdf_count = 0
        source_df = None
    else:
        # Incremental: read only CDF changes from silver.contacts
        start_ver = last_source_ver + 1
        try:
            cdf_df = (spark.read
                .format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_ver)
                .table(f"{sv}.contacts")
                .filter(f"tenant = {tenant_id}")
                .filter("_change_type IN ('insert', 'update_postimage')")
            )
            cdf_count = cdf_df.count()
            print(f"  CDF records found: {cdf_count} (versions {start_ver}..{source_max_ver})")
            
            if cdf_count == 0:
                print(f"  No changes detected — skipping MERGE")
                source_df = None
            else:
                source_df = cdf_df.select(
                    "id", "tenant", "email", "domain", "source_system", "source_system_object",
                    "source_key_name", "source_key_value", "data_timestamp", "a_accountid"
                )
        except Exception as cdf_err:
            # CDF not available or error — fall back to watermark-based filtering
            print(f"  CDF read failed ({cdf_err}), falling back to watermark-based filter")
            last_ts = spark.sql(f"""
                SELECT COALESCE(MAX(data_timestamp), TIMESTAMP('1900-01-01'))
                FROM {sv}.contacts_all WHERE tenant = {tenant_id}
            """).collect()[0][0]
            source_df = spark.sql(f"""
                SELECT id, tenant, email, domain, source_system, source_system_object,
                       source_key_name, source_key_value, data_timestamp, a_accountid
                FROM {sv}.contacts
                WHERE id IS NOT NULL AND tenant = {tenant_id}
                  AND data_timestamp > '{last_ts}'
            """)
            cdf_count = source_df.count()
            print(f"  Watermark-based records found: {cdf_count}")

    # Only run MERGE if we have changes
    if source_df is not None and cdf_count > 0:
        source_df.createOrReplaceTempView("contacts_cdf_source")
        
        spark.sql(f"""
            MERGE INTO {sv}.contacts_all AS target
            USING contacts_cdf_source AS source
            ON target.id = source.id AND target.tenant = source.tenant
            WHEN MATCHED AND source.data_timestamp > target.data_timestamp THEN UPDATE SET
                target.email = source.email,
                target.domain = source.domain,
                target.source_system = source.source_system,
                target.source_system_object = source.source_system_object,
                target.source_key_name = source.source_key_name,
                target.source_key_value = source.source_key_value,
                target.data_timestamp = source.data_timestamp,
                target.a_accountid = source.a_accountid
            WHEN NOT MATCHED THEN INSERT (id, tenant, email, domain, source_system,
                source_system_object, source_key_name, source_key_value, data_timestamp, a_accountid)
                VALUES (source.id, source.tenant, source.email, source.domain, source.source_system,
                    source.source_system_object, source.source_key_name, source.source_key_value,
                    source.data_timestamp, source.a_accountid)
        """)
        print(f"  MERGE complete: {cdf_count} CDF records processed")
    else:
        print(f"  No changes to process")

    # ── Save the source version we just processed ──
    spark.sql(f"ALTER TABLE {sv}.contacts_all SET TBLPROPERTIES ('last_source_cdf_version' = '{source_max_ver}')")
    print(f"  Saved last_source_cdf_version = {source_max_ver}")

except Exception as e:
    print(f"❌ contacts_all build failed: {e}")
    log_audit(
        job_name       = "02b_map_01_contacts_all",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise

In [0]:
# CELL 4 ── Verify and report
try:
    n = cnt(f"{sv}.contacts_all")
    print(f"  contacts_all: {n:,} rows built")

    # Spec DQ gate: no null IDs
    null_ids = spark.sql(f"SELECT COUNT(*) FROM {sv}.contacts_all WHERE id IS NULL").collect()[0][0]
    if null_ids > 0:
        print(f"  WARNING: {null_ids} null IDs in contacts_all")
    else:
        print(f"  DQ OK: no null IDs")
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_01_contacts_all",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise